In [ ]:
# dataset/LA_train/LA_train.zip
!cp /content/drive/MyDrive/lvths/dataset/LA_train/LA_train.zip /content/
!unzip -q /content/LA_train.zip -d /content/LA_train_data/
# -q là chế độ quiet, giúp Colab không in ra hàng chục ngàn dòng giải nén gây đơ trình duyệt

In [ ]:
# Cập nhật đường dẫn tương ứng với thư mục bạn vừa giải nén
PROTOCOL_PATH = "/content/LA_train_data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
AUDIO_DIR = "/content/LA_train_data/LA/ASVspoof2019_LA_train/flac/"

In [ ]:
!cp /content/drive/MyDrive/lvths/model/robust_aasist_student_best.pth /content/

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
import json
import numpy as np
from sklearn.metrics import roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d


In [ ]:
# --- THÊM HÀM COMPUTE EER CHUẨN ---
def compute_eer(y_true, y_scores):
    """Tính toán Equal Error Rate (EER) chuẩn cho bài toán Anti-spoofing"""
    fpr, tpr, thresholds = roc_curve(y_true, y_scores, pos_label=1)
    eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    return eer * 100 # Trả về phần trăm (%)

# --- CÁC THUẬT TOÁN TẤN CÔNG ---
def fgsm_attack(model, waveforms, labels, epsilon=0.005):
    waveforms_adv = waveforms.clone().detach().requires_grad_(True)
    logits, _ = model(waveforms_adv)
    loss = F.cross_entropy(logits, labels)
    model.zero_grad()
    loss.backward()
    perturbed = waveforms_adv + epsilon * waveforms_adv.grad.data.sign()
    return torch.clamp(perturbed, -1.0, 1.0).detach()

def pgd_attack(model, waveforms, labels, epsilon=0.005, alpha=0.001, iters=5):
    waveforms_adv = waveforms.clone().detach()
    waveforms_adv = waveforms_adv + torch.empty_like(waveforms_adv).uniform_(-epsilon, epsilon)
    waveforms_adv = torch.clamp(waveforms_adv, -1.0, 1.0)

    for _ in range(iters):
        waveforms_adv.requires_grad_(True)
        logits, _ = model(waveforms_adv)
        loss = F.cross_entropy(logits, labels)
        model.zero_grad()
        loss.backward()

        adv_data = waveforms_adv + alpha * waveforms_adv.grad.sign()
        eta = torch.clamp(adv_data - waveforms, min=-epsilon, max=epsilon)
        waveforms_adv = torch.clamp(waveforms + eta, min=-1.0, max=1.0).detach()
    return waveforms_adv

In [ ]:
# ==========================================
# 2. ĐỊNH NGHĨA DATASET
# ==========================================
class ASVspoof2019LA(Dataset):
    def __init__(self, protocol_file, audio_dir, max_frames=64000, target_sr=16000):
        self.audio_dir = audio_dir
        self.max_frames = max_frames
        self.target_sr = target_sr
        self.data_list = []
        with open(protocol_file, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    label = 0 if parts[4] == 'bonafide' else 1
                    self.data_list.append((parts[1], label))

    def __len__(self): return len(self.data_list)

    def process_audio(self, waveform, sr):
        if sr != self.target_sr:
            waveform = torchaudio.transforms.Resample(sr, self.target_sr)(waveform)
        num_frames = waveform.shape[1]
        if num_frames > self.max_frames:
            waveform = waveform[:, :self.max_frames]
        elif num_frames < self.max_frames:
            waveform = F.pad(waveform, (0, self.max_frames - num_frames))
        return waveform

    def __getitem__(self, idx):
        audio_id, label = self.data_list[idx]
        file_path = os.path.join(self.audio_dir, f"{audio_id}.flac")
        waveform, sr = torchaudio.load(file_path)
        if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)
        waveform = self.process_audio(waveform, sr).squeeze(0)
        return waveform, torch.tensor(label, dtype=torch.long)

In [ ]:
# ==========================================
# KHỞI TẠO ĐÁNH GIÁ (EVALUATION PIPELINE)
# ==========================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Bắt đầu đánh giá trên: {device}")

    # 1. KHỞI TẠO DATASET (TẬP EVAL CỦA BẠN)
    # LƯU Ý: Thay đổi đường dẫn này trỏ tới tập Eval (chứ không phải tập Train nữa)
    # PROTOCOL_PATH = "/content/LA_eval_data/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt"
    # AUDIO_DIR = "/content/LA_eval_data/ASVspoof2019_LA_eval/flac/"

    # Import class ASVspoof2019LA từ script trước của bạn
    eval_dataset = ASVspoof2019LA(PROTOCOL_PATH, AUDIO_DIR)
    eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False, num_workers=8)

    # 2. KHỞI TẠO 2 MÔ HÌNH
    # Import class RealAASIST của bạn
    CONFIG_PATH = "/content/drive/MyDrive/lvths/aasist_baseline/config/AASIST.conf"
    with open(CONFIG_PATH, "r") as f:
        model_config = json.load(f)["model_config"]

    model_baseline = RealAASIST(model_config).to(device)
    model_distilled = RealAASIST(model_config).to(device)

    # Tải trọng số (Thay đường dẫn thực tế của bạn)
    print("Đang tải trọng số...")
    model_baseline.load_state_dict(torch.load("/content/drive/MyDrive/baseline_aasist_fgsm.pth", map_location=device))
    model_distilled.load_state_dict(torch.load("/content/drive/MyDrive/robust_aasist_student.pth", map_location=device))

    model_baseline.eval()
    model_distilled.eval()

    # Lưu trữ điểm số
    results = {
        "baseline": {"clean": ([], []), "fgsm": ([], []), "pgd": ([], [])},
        "distilled": {"clean": ([], []), "fgsm": ([], []), "pgd": ([], [])}
    }

    # 3. VÒNG LẶP ĐÁNH GIÁ
    for waveforms, labels in tqdm(eval_loader, desc="Evaluating Models"):
        waveforms, labels = waveforms.to(device), labels.to(device)

        # Tạo dữ liệu tấn công (Dùng model_baseline làm mục tiêu sinh nhiễu để công bằng)
        with torch.enable_grad():
            adv_fgsm = fgsm_attack(model_baseline, waveforms, labels)
            adv_pgd = pgd_attack(model_baseline, waveforms, labels)

        with torch.no_grad():
            # --- ĐÁNH GIÁ BASELINE ---
            logits_clean_b, _ = model_baseline(waveforms)
            logits_fgsm_b, _ = model_baseline(adv_fgsm)
            logits_pgd_b, _ = model_baseline(adv_pgd)

            # --- ĐÁNH GIÁ DISTILLED ---
            logits_clean_d, _ = model_distilled(waveforms)
            logits_fgsm_d, _ = model_distilled(adv_fgsm)
            logits_pgd_d, _ = model_distilled(adv_pgd)

            # Lấy xác suất spoof (class 1)
            probs_clean_b = F.softmax(logits_clean_b, dim=1)[:, 1].cpu().numpy()
            probs_fgsm_b = F.softmax(logits_fgsm_b, dim=1)[:, 1].cpu().numpy()
            probs_pgd_b = F.softmax(logits_pgd_b, dim=1)[:, 1].cpu().numpy()

            probs_clean_d = F.softmax(logits_clean_d, dim=1)[:, 1].cpu().numpy()
            probs_fgsm_d = F.softmax(logits_fgsm_d, dim=1)[:, 1].cpu().numpy()
            probs_pgd_d = F.softmax(logits_pgd_d, dim=1)[:, 1].cpu().numpy()

            lbls = labels.cpu().numpy()

            # Lưu lại
            results["baseline"]["clean"][0].extend(lbls); results["baseline"]["clean"][1].extend(probs_clean_b)
            results["baseline"]["fgsm"][0].extend(lbls); results["baseline"]["fgsm"][1].extend(probs_fgsm_b)
            results["baseline"]["pgd"][0].extend(lbls); results["baseline"]["pgd"][1].extend(probs_pgd_b)

            results["distilled"]["clean"][0].extend(lbls); results["distilled"]["clean"][1].extend(probs_clean_d)
            results["distilled"]["fgsm"][0].extend(lbls); results["distilled"]["fgsm"][1].extend(probs_fgsm_d)
            results["distilled"]["pgd"][0].extend(lbls); results["distilled"]["pgd"][1].extend(probs_pgd_d)

    # 4. IN KẾT QUẢ TỔNG HỢP
    print("\n" + "="*50)
    print("📊 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ (EER %)")
    print("="*50)

    for model_name in ["baseline", "distilled"]:
        print(f"\n--- Mô hình: {model_name.upper()} ---")
        for attack_type in ["clean", "fgsm", "pgd"]:
            y_true = np.array(results[model_name][attack_type][0])
            y_scores = np.array(results[model_name][attack_type][1])
            eer_val = compute_eer(y_true, y_scores)
            print(f"👉 Dữ liệu {attack_type.upper():<5} : {eer_val:.3f}%")